# CardioSafe AI - Notebook 01: PTB-XL data preparation

Goal: download PTB-XL, build 5-superclass labels, use the official folds (train 1-8, val 9, test 10), explore the data, and cache arrays to Google Drive.

Runtime: any (CPU is fine for this notebook). Expect about 10-15 minutes total.

In [ ]:
!pip -q install wfdb neurokit2 lightgbm xgboost
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = '/content/drive/MyDrive/CardioSafe'
os.makedirs(WORK, exist_ok=True)
print('Work dir:', WORK)

In [ ]:
%%bash
cd /content
if [ ! -d ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3 ]; then
  wget -q --show-progress -O ptbxl.zip https://physionet.org/static/published-projects/ptb-xl/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3.zip
  unzip -q ptbxl.zip
fi
ls /content

In [ ]:
import glob, ast
import numpy as np, pandas as pd, wfdb
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

ROOT = glob.glob('/content/ptb-xl-*/')[0]
df = pd.read_csv(ROOT + 'ptbxl_database.csv', index_col='ecg_id')
df['scp_codes'] = df['scp_codes'].apply(ast.literal_eval)
agg = pd.read_csv(ROOT + 'scp_statements.csv', index_col=0)
print('Records:', df.shape[0])
df.head(3)

## Labels: 5 diagnostic superclasses (multi-label)

In [ ]:
diag = agg[agg['diagnostic'] == 1]
CLASSES = ['NORM', 'MI', 'STTC', 'CD', 'HYP']

def to_super(codes):
    out = set()
    for k in codes:
        if k in diag.index:
            out.add(diag.loc[k, 'diagnostic_class'])
    return sorted(out)

df['superclass'] = df['scp_codes'].apply(to_super)
Y_all = np.array([[c in s for c in CLASSES] for s in df['superclass']]).astype(np.float32)
has_label = Y_all.sum(1) > 0
print('Records with at least one superclass label:', has_label.sum(), 'of', len(df))
print('Dropping', (~has_label).sum(), 'records with no diagnostic superclass')

## Exploration

In [ ]:
d = df[has_label]
Y = Y_all[has_label]

print('Class counts (multi-label):')
print(pd.Series(Y.sum(0), index=CLASSES).astype(int))
print('\nLabels per record:', pd.Series(Y.sum(1)).value_counts().sort_index().to_dict())
print('\nRecords per fold:', d['strat_fold'].value_counts().sort_index().to_dict())
print('\nSex (0=male,1=female):', d['sex'].value_counts().to_dict())
print('Age: mean %.1f, min %d, max %d' % (d['age'].mean(), d['age'].min(), d['age'].max()))

print('\nPer-class counts in train/val/test:')
for name, mask in [('train', d['strat_fold'] <= 8), ('val', d['strat_fold'] == 9), ('test', d['strat_fold'] == 10)]:
    print(name, dict(zip(CLASSES, Y[mask.values].sum(0).astype(int))))

### Is there enough long-QT data for a dedicated QT classifier?
This decides how we build the QT part of the combined module.

In [ ]:
all_codes = pd.Series([k for c in df['scp_codes'] for k in c]).value_counts()
print('LNGQT (long QT) count:', int(all_codes.get('LNGQT', 0)))
print('\nRhythm / other codes of interest:')
for k in ['SBRAD', 'STACH', 'AFIB', '1AVB', '2AVB', '3AVB', 'LNGQT', 'LAFB', 'IRBBB', 'CRBBB', 'CLBBB']:
    print(f'{k:6s}', int(all_codes.get(k, 0)))
print('\nTop 25 SCP codes overall:')
print(all_codes.head(25))

## Load 100 Hz signals (about 21.8k records, takes several minutes)

In [ ]:
files = d['filename_lr'].values
X = np.stack([wfdb.rdsamp(ROOT + f)[0] for f in tqdm(files)]).astype(np.float32)
print('X shape:', X.shape, '(records, samples, leads)')
print('NaNs:', int(np.isnan(X).sum()))

In [ ]:
LEADS = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
fig, axes = plt.subplots(4, 1, figsize=(12, 8), sharex=True)
ids = [int(np.where(Y[:, i] == 1)[0][0]) for i in range(4)]
for ax, i, c in zip(axes, ids, CLASSES[:4]):
    ax.plot(X[i, :, 1])
    ax.set_ylabel(f'{c}\nlead II')
axes[-1].set_xlabel('samples at 100 Hz (10 s)')
plt.tight_layout(); plt.show()

## Cache to Drive

In [ ]:
fold = d['strat_fold'].values.astype(np.int8)
np.savez_compressed(
    WORK + '/ptbxl_100hz_super5.npz',
    X=X, Y=Y, fold=fold,
    ecg_id=d.index.values,
    age=d['age'].values.astype(np.float32),
    sex=d['sex'].values.astype(np.int8),
)
d.drop(columns=['scp_codes']).assign(superclass=d['superclass'].apply(','.join)).to_csv(WORK + '/ptbxl_meta_super5.csv')
print('Saved to', WORK)
print(os.listdir(WORK))